# Test

In [2]:
print("Hello, World!")

Hello, World!


In [3]:
dummy_data = """
{
  "programmes": [
    {
      "programme_name": "Computer Science",
      "subject_ids": [
        "CS101", "CS102", "CS103", "CS104", "CS105",
        "CS201", "CS202", "CS203",
        "SE201"
      ]
    },
    {
      "programme_name": "Software Engineering",
      "subject_ids": [
        "SE201", "SE202", "SE203", "SE204", "SE205", "SE206",
        "CS102", "CS203"
      ]
    },
    {
      "programme_name": "Intelligent System",
      "subject_ids": [
        "IS301", "IS302", "IS303", "IS304",
        "CS104", "CS201", "SE202"
      ]
    }
  ],
  "classrooms": {
    "assigned": [
      { "subject_id": "CS101", "classroom_id": "CR101" },
      { "subject_id": "CS102", "classroom_id": "CR102" },
      { "subject_id": "CS103", "classroom_id": "CR103" },
      { "subject_id": "SE201", "classroom_id": "CR104" },
      { "subject_id": "SE202", "classroom_id": "CR105" },
      { "subject_id": "IS301", "classroom_id": "CR106" },
      { "subject_id": "CS203", "classroom_id": "CR107" },
      { "subject_id": "CS104", "classroom_id": "CR108" }
    ],
    "unassigned": [
      "CR201",
      "CR202",
      "CR203",
      "CR204",
      "CR205",
      "CR206"
    ]
  },
  "lecturers": {
    "assigned": [
      { "subject_id": "CS101", "lecturer_id": "L001" },
      { "subject_id": "CS103", "lecturer_id": "L002" },
      { "subject_id": "SE201", "lecturer_id": "L003" },
      { "subject_id": "SE203", "lecturer_id": "L004" },
      { "subject_id": "IS302", "lecturer_id": "L005" },
      { "subject_id": "CS105", "lecturer_id": "L006" },
      { "subject_id": "SE204", "lecturer_id": "L007" },
      { "subject_id": "CS202", "lecturer_id": "L008" }
    ],
    "unassigned": [
      "L009",
      "L010",
      "L011",
      "L012"
    ]
  }
}
"""

import json
def load_dummy_data():
    return json.loads(dummy_data.strip())

data = load_dummy_data()

In [4]:
data

{'programmes': [{'programme_name': 'Computer Science',
   'subject_ids': ['CS101',
    'CS102',
    'CS103',
    'CS104',
    'CS105',
    'CS201',
    'CS202',
    'CS203',
    'SE201']},
  {'programme_name': 'Software Engineering',
   'subject_ids': ['SE201',
    'SE202',
    'SE203',
    'SE204',
    'SE205',
    'SE206',
    'CS102',
    'CS203']},
  {'programme_name': 'Intelligent System',
   'subject_ids': ['IS301',
    'IS302',
    'IS303',
    'IS304',
    'CS104',
    'CS201',
    'SE202']}],
 'classrooms': {'assigned': [{'subject_id': 'CS101', 'classroom_id': 'CR101'},
   {'subject_id': 'CS102', 'classroom_id': 'CR102'},
   {'subject_id': 'CS103', 'classroom_id': 'CR103'},
   {'subject_id': 'SE201', 'classroom_id': 'CR104'},
   {'subject_id': 'SE202', 'classroom_id': 'CR105'},
   {'subject_id': 'IS301', 'classroom_id': 'CR106'},
   {'subject_id': 'CS203', 'classroom_id': 'CR107'},
   {'subject_id': 'CS104', 'classroom_id': 'CR108'}],
  'unassigned': ['CR201', 'CR202', 'CR203'

In [5]:
data['programmes']

[{'programme_name': 'Computer Science',
  'subject_ids': ['CS101',
   'CS102',
   'CS103',
   'CS104',
   'CS105',
   'CS201',
   'CS202',
   'CS203',
   'SE201']},
 {'programme_name': 'Software Engineering',
  'subject_ids': ['SE201',
   'SE202',
   'SE203',
   'SE204',
   'SE205',
   'SE206',
   'CS102',
   'CS203']},
 {'programme_name': 'Intelligent System',
  'subject_ids': ['IS301',
   'IS302',
   'IS303',
   'IS304',
   'CS104',
   'CS201',
   'SE202']}]

In [1]:
def transform_json_to_csp(input):
    csp_data = {
        "variables": [],
        "domains": {
            "classrooms": [],
            "lecturers": []
        },
        "constraints": [],
    }

    # extract classrooms and lecturers

    classrooms = [classroom["classroom_id"] for classroom in input["classrooms"]["assigned"]] + input["classrooms"]["unassigned"]
    lecturers = [lecturer["lecturer_id"] for lecturer in input["lecturers"]["assigned"]] + input["lecturers"]["unassigned"]

    csp_data["domains"]["classrooms"] = list(set(classrooms))
    csp_data["domains"]["lecturers"] = list(set(lecturers))

    # build subject->classroom and subject->lecturer mappings
    classroom_mapping = {a["subject_id"]: a["classroom_id"] for a in input["classrooms"]["assigned"]}
    lecturer_mapping = {a["subject_id"]: a["lecturer_id"] for a in input["lecturers"]["assigned"]}
    
    # create variables for each subject
    subjects = set()
    subjects = set()
    for programme in input["programmes"]:
        subjects.update(programme["subject_ids"])

    
    for subject in subjects:
        # Determine classroom domain
        classroom_domain = [classroom_mapping[subject]] if subject in classroom_mapping else csp_data["domains"]["classrooms"]
        
        # Determine lecturer domain
        lecturer_domain = [lecturer_mapping[subject]] if subject in lecturer_mapping else csp_data["domains"]["lecturers"]

        var = {
            "name": subject,
            "domains": {
                "classroom": classroom_domain,
                "lecturer": lecturer_domain
            },
            "constraints": []
        }

        # Add hard constraints if they exist
        if subject in classroom_mapping:
            var["constraints"].append({
                "type": "hard",
                "field": "classroom",
                "value": classroom_mapping[subject]
            })
        
        if subject in lecturer_mapping:
            var["constraints"].append({
                "type": "hard",
                "field": "lecturer",
                "value": lecturer_mapping[subject]
            })

        csp_data["variables"].append(var)


    # add constraints for programmes
    for programme in input["programmes"]:
        subjects = programme["subject_ids"]
        if len(subjects) > 1:
            csp_data["constraints"].append({
               "type": "all_diff",
               "variables": subjects,
               "field": "classroom"
            })

    return csp_data

# dummy_data = data
csp_data = transform_json_to_csp(data)
print(json.dumps(csp_data, indent=2))

NameError: name 'data' is not defined

In [15]:
from constraint import Problem, AllDifferentConstraint
from itertools import combinations
from functools import partial
from collections import defaultdict
from typing import Dict, List, Set

In [16]:
from typing import Dict, List
class CSPScheduler:
    def __init__(self):
        self.timeslots = self._generate_timeslots()
    
    def _generate_timeslots(self)->List[str]:
        "Generate a list of 30-min timeslots for scheduling."
        days : List[str] = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
        hours = range(8, 18) # 8 AM to 5 PM
        timeslots: List[str] = []
        for hour in hours:
            timeslots.extend([f"{hour:02d}:00", f"{hour:02d}:30"])
        return [f"{day}_{time}" for day in days for time in timeslots]
    
    def transform_json_to_csp(self, input: Dict) -> Dict:
        """Optimized transformer with 30-min timeslot awareness"""
        classrooms: Set[str] = set()
        lecturers: Set[str] = set()
        # csp_data = {
        #     "variables": [],
        #     "domains": {
        #         "classrooms": list({
        #             classroom["classroom_id"] for classroom in input["classrooms"]["assigned"]
        #         }.union(input["classrooms"]["unassigned"])),
        #         "lecturers": list({
        #             lecturer["lecturer_id"] for lecturer in input["lecturers"]["assigned"]
        #         }.union(input["lecturers"]["unassigned"])),
        #     },
        #     "constraints": [],
        # }

        for classroom in input["classrooms"]["assigned"]:
            classrooms.add(classroom["classroom_id"])
        classrooms.update(input["classrooms"]["unassigned"])
        for lecturer in input["lecturers"]["assigned"]:
            lecturers.add(lecturer["lecturer_id"])
        lecturers.update(input["lecturers"]["unassigned"])

        # build subject->classroom and subject->lecturer mappings
        classroom_lookup = {a["subject_id"]: a["classroom_id"] \
                            for a in input["classrooms"]["assigned"]}
        lecturer_lookup = {a["subject_id"]: a["lecturer_id"] \
                           for a in input["lecturers"]["assigned"]}
        
        subjects = set()
        programme_map = defaultdict(list)
        duration_slots = 6 # Assuming each subject takes 3 hours, which is 6 half-hour slots
        
        for programme in input["programmes"]:
            for subject in programme["subject_ids"]:
                if subject not in subjects:
                    subjects.add(subject)
                    programme_map[programme["programme_name"]].append(subject)
                # var = {
                #     "name": subject,
                #     "duration_slots": duration_slots,
                #     "domains": {
                #         "classroom": [classroom_lookup[subject]] if subject in classroom_lookup else csp_data["domains"]["classrooms"],
                #         "lecturer": [lecturer_lookup[subject]] if subject in lecturer_lookup else csp_data["domains"]["lecturers"]
                #     },
                #     "constraints": []
                # }

                # if subject in classroom_lookup:
                #     var["constraints"].append({
                #         "type": "hard",
                #         "field": "classroom",
                #         "value": classroom_lookup[subject]
                #     })
                # if subject in lecturer_lookup:
                #     var["constraints"].append({
                #         "type": "hard",
                #         "field": "lecturer",
                #         "value": lecturer_lookup[subject]
                #     })
                
                # csp_data["variables"].append(var)
                # programme_map[programme["programme_name"]].append(subject)
        
        # create CSP data structure
        variables = []
        for subject in subjects:
            var = {
                "name": subject,
                "duration_slots": duration_slots,
                "domains": {
                    "classroom": [classroom_lookup[subject]] \
                        if subject in classroom_lookup else list(classrooms),
                    "lecturer": [lecturer_lookup[subject]] \
                        if subject in lecturer_lookup else list(lecturers)
                },
                "constraints": []
            }

            # Add hard constraints if they exist
            if subject in classroom_lookup:
                var["constraints"].append({
                    "type": "hard",
                    "field": "classroom",
                    "value": classroom_lookup[subject]
                })
            if subject in lecturer_lookup:
                var["constraints"].append({
                    "type": "hard",
                    "field": "lecturer",
                    "value": lecturer_lookup[subject]
                })
            
            variables.append(var)

        # Add programme constraints
        # for subjects in programme_map.values():
        #     if len(subjects) > 1:
        #         csp_data["constraints"].append({
        #             "type": "programme_conflict",
        #             "variables": subjects,
        #         })
        
        # return csp_data
        constraints = []
        for subjects in programme_map.values():
            if len(subjects) > 1:
                constraints.append({
                    "type": "programme_conflict",
                    "variables": subjects,
                })
        
        return {
            "variables": variables,
            "domains": {
                "classrooms": list(classrooms),
                "lecturers": list(lecturers)
            },
            "constraints": constraints,
        }
    
    def solve(self, csp_data: Dict) -> Dict:
        """Solve the CSP using the constraint library."""
        problem = Problem()
        var_mapping = {}
        
        # create variables in the problem
        # for var in csp_data["variables"]:
        #     subject = var["name"]

        #     # classroom variable
        #     room_var = f"{subject}_room"
        #     problem.addVariable(room_var, var["domains"]["classroom"])

        #     # timeslot variable
        #     for i in range(var["duration_slots"]):
        #         time_var = f"{subject}_time_{i}"
        #         problem.addVariable(time_var, self.timeslots)
        #         var_mapping.setdefault(subject, {"room": room_var, "times": []})["times"].append(time_var)
            
        #     time_vars = var_mapping[subject]["times"]
        #     if len(time_vars) > 1:
        #         problem.addConstraint(
        #         self._consecutive_slots_constraint,
        #         time_vars
        #         )
        for var in csp_data["variables"]:
            subject = var["name"]

            # classroom variable
            room_var = f"{subject}_room"
            problem.addVariable(room_var, var["domains"]["classroom"])
            # var_mapping[subject] = {"room": room_var, "times": []}

            # timeslot variables
            time_vars = []
            for i in range(var["duration_slots"]):
                time_var = f"{subject}_time_{i}"
                problem.addVariable(time_var, self.timeslots)
                time_vars.append(time_var)

            var_mapping[subject] = {"room": room_var, "times": time_vars}
            # time_vars = var_mapping[subject]["times"]

            # add consecutive slots constraint if more than one timeslot
            if len(time_vars) > 1:
                problem.addConstraint(
                    self._consecutive_slots_constraint,
                    time_vars
                )

        # add constraints
        self._add_base_constraints(problem, csp_data, var_mapping)

        # solve the problem & format
        solution  = problem.getSolution()
        if not solution:
            raise ValueError("No valid schedule found.")
        return self._format_solution(solution, csp_data, var_mapping)
    
    def _consecutive_slots_constraint(self, *slots):
        """Ensure that the timeslots are consecutive."""
        sorted_slots = sorted(slots)
        for i in range(1, len(slots)):
            current_day, current_time = sorted_slots[i].split('_')
            prev_day, prev_time = sorted_slots[i-1].split('_')

            # same day
            if current_day != prev_day:
                return False
            
            # current time is 30 minutes after previous time
            curr_h, curr_m = map(int, current_time.split(":"))
            prev_h, prev_m = map(int, prev_time.split(":"))
            
            # Check 30-minute increment
            if not ((curr_h == prev_h and curr_m == prev_m + 30) or
                   (curr_h == prev_h + 1 and curr_m == 0 and prev_m == 30)):
                return False

        return True
    
    def _add_base_constraints(self, problem, csp_data, var_mapping):
        """Add base constraints to the CSP problem."""
        # classroom assignments 
        for var in csp_data["variables"]:
            for constraint in var["constraints"]:
                if constraint["field"] == "classroom":
                    problem.addConstraint(
                        lambda value, c=constraint: value == c["value"],
                        [var_mapping[var["name"]]["room"]],
                    )

        # programme conflicts
        for constraint in csp_data["constraints"]:
            if constraint["type"] == "programme_conflict":
                all_times = []
                for subject in constraint["variables"]:
                    all_times.extend(var_mapping[subject]["times"])
                
                problem.addConstraint(
                    AllDifferentConstraint(),
                    all_times
                )
        
        # lecturer conflicts
        lecturer_map = defaultdict(list)
        for var in csp_data["variables"]:
            for constraint in var["constraints"]:
                if constraint["field"] == "lecturer":
                    lecturer_map[constraint["value"]].append(var_mapping[var["name"]]["times"])

        for times in lecturer_map.values():
            if len(times) > 1:
                problem.addConstraint(
                    AllDifferentConstraint(),
                    times
                )

    def _format_solution(self, solution, var_mapping):
        """Format the solution into a more readable structure."""
        schedule = {}
        for var in csp_data["variables"]:
            subject = var["name"]
            room = solution[var_mapping[subject]["room"]]
            times = sorted([solution[time_var] for time_var in var_mapping[subject]["times"]])
            
            start_time = times[0]
            # end_time = times[-1].split("_")
            end_day, end_time = times[-1].split("_")
            # end_time[1] = f"{int(end_time[1].split(':')[0]) + 1}:00"
            end_h = int(end_time.split(":")[0])
            end_time = f"{end_h+1}:00" if end_time.endswith("30") else f"{end_h}:30"


            lecturer = next(
                (constraint["value"] for constraint in var["constraints"] \
                 if constraint["field"] == "lecturer"),
                None
            )

            schedule[subject] = {
                "classroom": room,
                "lecturer": lecturer,
                "start_time": start_time,
                "end_time": "_".join(end_time),
                "duration" : f"{len(times)*30} minutes",
            }

        return schedule
    
    def run_pipeline(self, input_json: Dict) -> Dict:
        """Run the entire pipeline from JSON to CSP solution."""
        csp_data = self.transform_json_to_csp(input_json)
        return self.solve(csp_data)



In [17]:
scheduler = CSPScheduler()

try:
    schedule = scheduler.run_pipeline(data)
    print(json.dumps(schedule, indent=2))
except ValueError as e:
    print(f"Error: {e}")

KeyboardInterrupt: 

In [23]:
from typing import Dict, List, Set
from collections import defaultdict
from constraint import Problem, AllDifferentConstraint, MinConflictsSolver
import json

class CSPScheduler:
    def __init__(self):
        self.timeslots = self._generate_timeslots()
    
    def _generate_timeslots(self) -> List[str]:
        """Generate 30-minute timeslots from 8AM to 6PM, Monday-Friday"""
        days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
        hours = range(8, 18)  # 8AM to 5PM
        times = []
        for hour in hours:
            times.extend([f"{hour:02d}:00", f"{hour:02d}:30"])
        print("Generated timeslots")
        return [f"{day}_{time}" for day in days for time in times]
    
    def transform_json_to_csp(self, input_data: Dict) -> Dict:
        """Transform input JSON to CSP format with duplicate handling"""
        # Extract all unique classrooms and lecturers
        classrooms: Set[str] = set()
        lecturers: Set[str] = set()
        
        for classroom in input_data["classrooms"]["assigned"]:
            classrooms.add(classroom["classroom_id"])
        classrooms.update(input_data["classrooms"]["unassigned"])
        
        for lecturer in input_data["lecturers"]["assigned"]:
            lecturers.add(lecturer["lecturer_id"])
        lecturers.update(input_data["lecturers"]["unassigned"])
        
        # Build lookup dictionaries
        classroom_lookup = {a["subject_id"]: a["classroom_id"] 
                          for a in input_data["classrooms"]["assigned"]}
        lecturer_lookup = {a["subject_id"]: a["lecturer_id"] 
                         for a in input_data["lecturers"]["assigned"]}
        
        # Track all unique subjects
        subjects = set()
        programme_map = defaultdict(list)
        
        for programme in input_data["programmes"]:
            for subject in programme["subject_ids"]:
                if subject not in subjects:  # Only process each subject once
                    subjects.add(subject)
                    programme_map[programme["programme_name"]].append(subject)
        
        # Create CSP variables
        variables = []
        for subject in subjects:
            var = {
                "name": subject,
                # "duration_slots": 6,  # 3 hours = 6 * 30min slots
                "duration_slots": 6,  # 3 hours = 6 * 30min slots
                "domains": {
                    "classroom": [classroom_lookup[subject]] if subject in classroom_lookup 
                              else list(classrooms),
                    "lecturer": [lecturer_lookup[subject]] if subject in lecturer_lookup 
                             else list(lecturers)
                },
                "constraints": []
            }
            
            if subject in classroom_lookup:
                var["constraints"].append({
                    "type": "hard",
                    "field": "classroom",
                    "value": classroom_lookup[subject]
                })
                
            if subject in lecturer_lookup:
                var["constraints"].append({
                    "type": "hard",
                    "field": "lecturer",
                    "value": lecturer_lookup[subject]
                })
            
            variables.append(var)
        
        # Create programme constraints
        constraints = []
        for subjects in programme_map.values():
            if len(subjects) > 1:
                constraints.append({
                    "type": "programme_conflict",
                    "variables": subjects
                })
        print("Transformed JSON to CSP format")

        return {
            "variables": variables,
            "domains": {
                "classrooms": list(classrooms),
                "lecturers": list(lecturers)
            },
            "constraints": constraints
        }
    
    def solve(self, csp_data: Dict) -> Dict:
        """Solve the CSP problem with duplicate checking"""
        problem = Problem(MinConflictsSolver(500))
        var_mapping = {}
        
        # Create variables with unique names
        for var in csp_data["variables"]:
            subject = var["name"]
            
            # Classroom variable (unique per subject)
            room_var = f"{subject}_room"
            problem.addVariable(room_var, var["domains"]["classroom"])
            
            # Timeslot variables (multiple per subject)
            time_vars = []
            for i in range(var["duration_slots"]):
                time_var = f"{subject}_time_{i}"
                problem.addVariable(time_var, self.timeslots)
                time_vars.append(time_var)
            
            var_mapping[subject] = {
                "room": room_var,
                "times": time_vars
            }
            
            # Add consecutive slot constraint if needed
            if len(time_vars) > 1:
                problem.addConstraint(
                    self._consecutive_slots_constraint,
                    time_vars
                )
        
        # Add all constraints
        self._add_constraints(problem, csp_data, var_mapping)
        
        # Get solution
        solution = problem.getSolution()
        if not solution:
            raise ValueError("No valid schedule found")
        
        print("Solved CSP problem")
        
        return self._format_solution(solution, csp_data, var_mapping)
    
    def _consecutive_slots_constraint(self, *slots: str) -> bool:
        """Ensure timeslots are consecutive 30-minute intervals"""
        sorted_slots = sorted(slots)
        for i in range(1, len(sorted_slots)):
            curr_day, curr_time = sorted_slots[i].split('_')
            prev_day, prev_time = sorted_slots[i-1].split('_')
            
            # Must be same day
            if curr_day != prev_day:
                return False
                
            curr_h, curr_m = map(int, curr_time.split(':'))
            prev_h, prev_m = map(int, prev_time.split(':'))
            
            # Check 30-minute increment
            if not ((curr_h == prev_h and curr_m == prev_m + 30) or
                   (curr_h == prev_h + 1 and curr_m == 0 and prev_m == 30)):
                return False
        
        print("Checked consecutive slots constraint")
                
        return True
    
    def _add_constraints(self, problem: Problem, csp_data: Dict, var_mapping: Dict):
        """Add all constraints to the problem"""
        # Classroom assignments
        for var in csp_data["variables"]:
            for constraint in var["constraints"]:
                if constraint["field"] == "classroom":
                    problem.addConstraint(
                        lambda value, c=constraint: value == c["value"],
                        [var_mapping[var["name"]]["room"]]
                    )
        
        # Programme conflicts (no overlapping times)
        for constraint in csp_data["constraints"]:
            if constraint["type"] == "programme_conflict":
                all_times = []
                for subject in constraint["variables"]:
                    all_times.extend(var_mapping[subject]["times"])
                problem.addConstraint(AllDifferentConstraint(), all_times)
        
        # Lecturer conflicts
        lecturer_map = defaultdict(list)
        for var in csp_data["variables"]:
            for constraint in var["constraints"]:
                if constraint["field"] == "lecturer":
                    lecturer_map[constraint["value"]].extend(var_mapping[var["name"]]["times"])
        
        for times in lecturer_map.values():
            if len(times) > 1:
                problem.addConstraint(AllDifferentConstraint(), times)

        print("Added all constraints to the problem")
    
    def _format_solution(self, solution: Dict, csp_data: Dict, var_mapping: Dict) -> Dict:
        """Format the solution into a readable schedule"""
        schedule = {}
        for var in csp_data["variables"]:
            subject = var["name"]
            room = solution[var_mapping[subject]["room"]]
            time_slots = sorted([solution[t] for t in var_mapping[subject]["times"]])
            
            start = time_slots[0]
            end_day, end_time = time_slots[-1].split('_')
            end_h = int(end_time.split(':')[0])
            end_time = f"{end_h + 1}:00" if not end_time.endswith('30') else f"{end_h}:30"
            
            lecturer = next(
                (c["value"] for c in var["constraints"] if c["field"] == "lecturer"),
                None
            )
            
            schedule[subject] = {
                "classroom": room,
                "lecturer": lecturer,
                "start": start,
                "end": f"{end_day}_{end_time}",
                "duration": f"{len(time_slots)*30} minutes"
            }

        print("Formatted solution into schedule")
        return schedule
    
    def run_pipeline(self, input_json: Dict) -> Dict:
        """Complete scheduling pipeline"""
        csp_data = self.transform_json_to_csp(input_json)
        return self.solve(csp_data)

# Usage example
if __name__ == "__main__":
    scheduler = CSPScheduler()
    
    # with open('input.json') as f:
    #     data = json.load(f)
    
    try:
        schedule = scheduler.run_pipeline(data)
        print(json.dumps(schedule, indent=2))
    except ValueError as e:
        print(f"Error: {e}")

Generated timeslots
Transformed JSON to CSP format
Added all constraints to the problem
Error: No valid schedule found


In [30]:
from typing import Dict, List, Set, DefaultDict
from collections import defaultdict
from constraint import Problem, AllDifferentConstraint
# import json

class CSPScheduler:
    def __init__(self):
        self.timeslots = self._generate_timeslots()
        self.classroom_availability = defaultdict(set)  # Track classroom usage
    
    def _generate_timeslots(self) -> List[str]:
        """Generate 30-minute timeslots from 8AM to 6PM, Monday-Friday"""
        days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
        hours = range(8, 18)  # 8AM to 5PM (last slot starts at 5:00)
        times = []
        for hour in hours:
            times.extend([f"{hour:02d}:00", f"{hour:02d}:30"])
        return [f"{day}_{time}" for day in days for time in times]
    
    def _is_classroom_available(self, classroom: str, required_slots: List[str]) -> bool:
        """Check if classroom is available for all required slots"""
        return all(slot not in self.classroom_availability[classroom] 
                 for slot in required_slots)
    
    def solve(self, csp_data: Dict) -> Dict:
        """Solve with classroom reuse"""
        problem = Problem()
        var_mapping = {}
        
        # 1. Create variables
        for var in csp_data["variables"]:
            subject = var["name"]
            
            # Classroom variable
            room_var = f"{subject}_room"
            problem.addVariable(room_var, var["domains"]["classroom"])
            
            # Timeslot variables (first slot only)
            time_var = f"{subject}_time"
            problem.addVariable(time_var, self.timeslots)
            
            var_mapping[subject] = {
                "room": room_var,
                "time": time_var,
                "duration": var["duration_slots"]
            }
        
        # 2. Add constraints
        # Classroom assignments (hard constraints)
        for var in csp_data["variables"]:
            for constraint in var["constraints"]:
                if constraint["field"] == "classroom":
                    problem.addConstraint(
                        lambda value, c=constraint: value == c["value"],
                        [var_mapping[var["name"]]["room"]]
                    )
        
        # Programme conflicts
        for constraint in csp_data["constraints"]:
            if constraint["type"] == "programme_conflict":
                problem.addConstraint(
                    AllDifferentConstraint(),
                    [var_mapping[var]["time"] for var in constraint["variables"]]
                )
        
        # Lecturer conflicts
        lecturer_map = defaultdict(list)
        for var in csp_data["variables"]:
            for constraint in var["constraints"]:
                if constraint["field"] == "lecturer":
                    lecturer_map[constraint["value"]].append(var_mapping[var["name"]]["time"])
        
        for times in lecturer_map.values():
            if len(times) > 1:
                problem.addConstraint(AllDifferentConstraint(), times)
        
        # 3. Custom constraint for classroom availability
        def classroom_constraint(room, start_time, duration):
            """Ensure classroom isn't double-booked"""
            day, time = start_time.split('_')
            hour, minute = map(int, time.split(':'))
            
            required_slots = []
            for i in range(duration):
                if minute == 30:
                    hour += 1
                    minute = 0
                else:
                    minute = 30
                required_slots.append(f"{day}_{hour:02d}:{minute:02d}")
            
            return self._is_classroom_available(room, required_slots)
        
        for subject, data in var_mapping.items():
            problem.addConstraint(
                lambda r, t, d=data["duration"]: classroom_constraint(r, t, d),
                [data["room"], data["time"]]
            )
        
        # 4. Solve
        solution = problem.getSolution()
        if not solution:
            raise ValueError("No valid schedule found")
        
        # 5. Update classroom availability
        for subject, data in var_mapping.items():
            room = solution[data["room"]]
            start_time = solution[data["time"]]
            day, time = start_time.split('_')
            hour, minute = map(int, time.split(':'))
            
            for i in range(data["duration"]):
                slot = f"{day}_{hour:02d}:{minute:02d}"
                self.classroom_availability[room].add(slot)
                if minute == 30:
                    hour += 1
                    minute = 0
                else:
                    minute = 30
        
        return self._format_solution(solution, csp_data, var_mapping)
    
    def _format_solution(self, solution: Dict, csp_data: Dict, var_mapping: Dict) -> Dict:
        """Format the final schedule"""
        schedule = {}
        for var in csp_data["variables"]:
            subject = var["name"]
            room = solution[var_mapping[subject]["room"]]
            start_time = solution[var_mapping[subject]["time"]]
            
            # Calculate end time
            day, time = start_time.split('_')
            hour, minute = map(int, time.split(':'))
            duration = var_mapping[subject]["duration"]
            
            # Advance by duration
            for _ in range(duration):
                if minute == 30:
                    hour += 1
                    minute = 0
                else:
                    minute = 30
            
            end_time = f"{day}_{hour:02d}:{minute:02d}"
            
            lecturer = next(
                (c["value"] for c in var["constraints"] 
                 if c["field"] == "lecturer"),
                None
            )
            
            schedule[subject] = {
                "classroom": room,
                "lecturer": lecturer,
                "start": start_time,
                "end": end_time,
                "duration": f"{duration*30} minutes"
            }
        return schedule
    
    def transform_json_to_csp(self, input_data: Dict) -> Dict:
        """Transform input JSON to CSP format with duplicate handling"""
        # Extract all unique classrooms and lecturers
        classrooms: Set[str] = set()
        lecturers: Set[str] = set()
        
        for classroom in input_data["classrooms"]["assigned"]:
            classrooms.add(classroom["classroom_id"])
        classrooms.update(input_data["classrooms"]["unassigned"])
        
        for lecturer in input_data["lecturers"]["assigned"]:
            lecturers.add(lecturer["lecturer_id"])
        lecturers.update(input_data["lecturers"]["unassigned"])
        
        # Build lookup dictionaries
        classroom_lookup = {a["subject_id"]: a["classroom_id"] 
                          for a in input_data["classrooms"]["assigned"]}
        lecturer_lookup = {a["subject_id"]: a["lecturer_id"] 
                         for a in input_data["lecturers"]["assigned"]}
        
        # Track all unique subjects
        subjects = set()
        programme_map = defaultdict(list)
        
        for programme in input_data["programmes"]:
            for subject in programme["subject_ids"]:
                if subject not in subjects:  # Only process each subject once
                    subjects.add(subject)
                    programme_map[programme["programme_name"]].append(subject)
        
        # Create CSP variables
        variables = []
        for subject in subjects:
            var = {
                "name": subject,
                # "duration_slots": 6,  # 3 hours = 6 * 30min slots
                "duration_slots": 6,  # 3 hours = 6 * 30min slots
                "domains": {
                    "classroom": [classroom_lookup[subject]] if subject in classroom_lookup 
                              else list(classrooms),
                    "lecturer": [lecturer_lookup[subject]] if subject in lecturer_lookup 
                             else list(lecturers)
                },
                "constraints": []
            }
            
            if subject in classroom_lookup:
                var["constraints"].append({
                    "type": "hard",
                    "field": "classroom",
                    "value": classroom_lookup[subject]
                })
                
            if subject in lecturer_lookup:
                var["constraints"].append({
                    "type": "hard",
                    "field": "lecturer",
                    "value": lecturer_lookup[subject]
                })
            
            variables.append(var)
        
        # Create programme constraints
        constraints = []
        for subjects in programme_map.values():
            if len(subjects) > 1:
                constraints.append({
                    "type": "programme_conflict",
                    "variables": subjects
                })
        print("Transformed JSON to CSP format")

        return {
            "variables": variables,
            "domains": {
                "classrooms": list(classrooms),
                "lecturers": list(lecturers)
            },
            "constraints": constraints
        }

    def run_pipeline(self, input_json: Dict) -> Dict:
        """Complete scheduling pipeline"""
        csp_data = self.transform_json_to_csp(input_json)
        return self.solve(csp_data)
    

csp = CSPScheduler()
# print(len(csp.run_pipeline(data)))
csp.run_pipeline(data)

# len(data["programmes"][0]["subject_ids"])

Transformed JSON to CSP format


{'CS102': {'classroom': 'CR102',
  'lecturer': None,
  'start': 'Friday_17:00',
  'end': 'Friday_20:00',
  'duration': '180 minutes'},
 'CS105': {'classroom': 'CR103',
  'lecturer': 'L006',
  'start': 'Friday_15:30',
  'end': 'Friday_18:30',
  'duration': '180 minutes'},
 'SE205': {'classroom': 'CR103',
  'lecturer': None,
  'start': 'Friday_16:00',
  'end': 'Friday_19:00',
  'duration': '180 minutes'},
 'SE206': {'classroom': 'CR103',
  'lecturer': None,
  'start': 'Friday_15:30',
  'end': 'Friday_18:30',
  'duration': '180 minutes'},
 'IS303': {'classroom': 'CR103',
  'lecturer': None,
  'start': 'Friday_16:30',
  'end': 'Friday_19:30',
  'duration': '180 minutes'},
 'CS203': {'classroom': 'CR107',
  'lecturer': None,
  'start': 'Friday_14:00',
  'end': 'Friday_17:00',
  'duration': '180 minutes'},
 'IS302': {'classroom': 'CR103',
  'lecturer': 'L005',
  'start': 'Friday_17:00',
  'end': 'Friday_20:00',
  'duration': '180 minutes'},
 'CS104': {'classroom': 'CR108',
  'lecturer': None

# Second Try

In [2]:
data_str = """
{
  
    "classrooms": [
      "Hall 1",
      "Lab A",
      "Lab C",
      "Lab E",
      "M001",
      "R201",
      "R202",
      "R203",
      "R301",
      "R302"
    ],
    "department": "Computing",
    "level": "Degree",
    "programmes": [
      "Bachelor of Computer Science"
    ],
    "subjects": [
      {
        "classroom_id": "Lab C",
        "duration": 3,
        "lecturer_id": "Mr David",
        "name": "Object-Oriented Programming",
        "subject_id": "CS004"
      },
      {
        "classroom_id": "Lab A",
        "duration": 2,
        "lecturer_id": "Low Jing Xuan",
        "name": "Database Systems",
        "subject_id": "CS005"
      },
      {
        "classroom_id": "R201",
        "duration": 3,
        "lecturer_id": "Ms. Sarah",
        "name": "Computer Architecture and Organization",
        "subject_id": "CS006"
      },
      {
        "classroom_id": "R301",
        "duration": 4,
        "lecturer_id": "Dr. Aisha Patel",
        "name": "Artificial Intelligence",
        "subject_id": "CS008"
      },
      {
        "classroom_id": "Hall 1",
        "duration": 4,
        "lecturer_id": "Dr. Aisha Patel",
        "name": "Cybersecurity and Cryptography",
        "subject_id": "CS010"
      },
      {
        "classroom_id": "Hall 1",
        "duration": 3,
        "lecturer_id": "Prof. Michael Chen",
        "name": "Cloud Computing",
        "subject_id": "CS012"
      },
      {
        "classroom_id": "Hall 1",
        "duration": 4,
        "lecturer_id": "Prof. Michael Chen",
        "name": "Mobile Application Development",
        "subject_id": "CS014"
      },
      {
        "duration": 3,
        "lecturer_id": "Prof. Michael Chen",
        "name": "Machine Learning Fundamentals",
        "subject_id": "CS003"
      },
      {
        "duration": 3.5,
        "lecturer_id": "Caeley Yong",
        "name": "Operating Systems",
        "subject_id": "CS007"
      },
      {
        "duration": 2,
        "lecturer_id": "Mr David",
        "name": "Web Technologies and Development",
        "subject_id": "CS009"
      },
      {
        "duration": 3,
        "lecturer_id": "Dr. Aisha Patel",
        "name": "Parallel and Distributed Systems",
        "subject_id": "CS011"
      },
      {
        "duration": 2,
        "lecturer_id": "Dr. Emma",
        "name": "Data Mining and Analytics",
        "subject_id": "CS013"
      },
      {
        "duration": 1,
        "lecturer_id": "Ms. Sarah",
        "name": "Final Year Project",
        "subject_id": "CS015"
      }
    ]
}
"""

In [3]:
import json

data = json.loads(data_str.strip())

data

{'classrooms': ['Hall 1',
  'Lab A',
  'Lab C',
  'Lab E',
  'M001',
  'R201',
  'R202',
  'R203',
  'R301',
  'R302'],
 'department': 'Computing',
 'level': 'Degree',
 'programmes': ['Bachelor of Computer Science'],
 'subjects': [{'classroom_id': 'Lab C',
   'duration': 3,
   'lecturer_id': 'Mr David',
   'name': 'Object-Oriented Programming',
   'subject_id': 'CS004'},
  {'classroom_id': 'Lab A',
   'duration': 2,
   'lecturer_id': 'Low Jing Xuan',
   'name': 'Database Systems',
   'subject_id': 'CS005'},
  {'classroom_id': 'R201',
   'duration': 3,
   'lecturer_id': 'Ms. Sarah',
   'name': 'Computer Architecture and Organization',
   'subject_id': 'CS006'},
  {'classroom_id': 'R301',
   'duration': 4,
   'lecturer_id': 'Dr. Aisha Patel',
   'name': 'Artificial Intelligence',
   'subject_id': 'CS008'},
  {'classroom_id': 'Hall 1',
   'duration': 4,
   'lecturer_id': 'Dr. Aisha Patel',
   'name': 'Cybersecurity and Cryptography',
   'subject_id': 'CS010'},
  {'classroom_id': 'Hall 1',

In [ ]:
from pydantic import BaseModel, Field
from typing import List
from app.models.base_model import Classroom, Subject

class Scheduler(BaseModel):
    hours_per_day: int # 9 a.m. to 5 p.m.
    working_days: int # Monday to Friday
    time_slots: list = []
    days_map: Dict[int, str] = {
        1: "Monday",
        2: "Tuesday", 
        3:"Wednesday",
        4: "Thursday",
        5: "Friday",
    }

    def slot_to_time(self, slot_index, slot_size=30, start=9):
        """
        Convert slot index -> HH:MM string.
        Example: slot_index=0 (9:00), slot_index=1 (9:30), ...
        """

        min = start * 30 + slot_index * slot_size
        h, m = divmod(min, 60)

        return f"{h:02d}:{m:02d}"

    def run(self, data):
        raise NotImplementedError("Scheduler interface is not implemented")

class Timetable(BaseModel):
    subjects: List[Subject] = Field(..., title="List of subjects")
    classrooms: List[Classroom] = Field(..., title="List of classrooms")
    model: Scheduler = Field(..., title="Scheduler technique")
    
    # allow Scheduler sub-classes to be passed
    class Config:
        arbitrary_types_allowed = True

    def generate(self):
        data = {
            "subjects": self.subjects,
            "classrooms": self.classrooms,
        }
        self.model.run(data=data)

In [33]:
# get classrooms and subject information
classrooms = list(data.get("classrooms")) or []
available_classrooms = [Classroom(classroom_id=c_id) for c_id in classrooms]

subjects = list(data.get("subjects")) or []
subjects = [
    Subject(
        classroom_id=Classroom(classroom_id=subject.get("classroom_id") or None),
        duration=subject.get("duration"),
        lecturer_id=subject.get("lecturer_id"),
        name=subject.get("name"),
        subject_id=subject.get("subject_id"),
    )
    for subject in subjects
]

# timetable = Timetable(subjects=subjects, classrooms=available_classrooms, model=CSPScheduler())
# timetable.generate()

In [ ]:
# from app.models.base_model import Scheduler
# from pydantic import Field
from app.utils.logger import logger
from constraint import Problem, RecursiveBacktrackingSolver
from typing import Any
import random
import pandas as pd
import copy

class CSPScheduler(Scheduler):


    solver : Any = None
    problem: Any = None
    
    # hours_per_day: int = Field(..., title="Number of working hours in a day") # 9 a.m. to 5 p.m.
    # working_days: int = Field(..., title="Number of working days in a week") # Monday to Friday
    # time_slots: list = Field(default=[])
    # days_map: Dict[int, str] = Field(default={
    #     1: "Monday",
    #     2: "Tuesday", 
    #     3:"Wednesday",
    #     4: "Thursday",
    #     5: "Friday",
    # })

    def parse(self, subjects: List[Subject]):
        subjects_as_dict = []
        for subject in subjects:
            sub = json.loads(subject.model_dump_json())
            sub["classroom_id"] = sub["classroom_id"]["classroom_id"]

            subjects_as_dict.append(sub)

        return subjects_as_dict

    def run(self, data):
        # logger.debug("CSP Model is running")
        self.solver = RecursiveBacktrackingSolver(forwardcheck=True)
        self.solver.orderDomainValues = True
        self.solver.random = True

        self.problem = Problem(solver=self.solver)

        self.time_slots =  [
            (day, slot) 
            for day in range(1, self.working_days+1)
            for slot in range(1, int(self.hours_per_day * 2)+1)
        ] # Create 30-min time slots

        # get the data
        classrooms = data.get("classrooms") or []
        subjects = data.get("subjects") or []

        # logger.info(f"classrooms: {classrooms}")
        # logger.info(f"subjects: {subjects}")
        # logger.info(f"timeslots: {self.time_slots}")

        self._create_domains(subjects=subjects, classrooms=classrooms)
        self._add_constraints(subjects=subjects)

        solution = self.problem.getSolution()
        logger.info(solution)

        result = copy.deepcopy(solution)

        subjects_as_dict = self.parse(subjects=subjects)        

        subjects_df = pd.DataFrame(subjects_as_dict)
        
        for subject_id, (day, start, room) in solution.items():
            sub = subjects_df[subjects_df["subject_id"] == subject_id]
            day = self.days_map[day]
            duration = sub["duration"].iloc[0]
            lecturer = sub["lecturer_id"].iloc[0]
            end_hrs = self.slot_to_time(start + int(duration * 2))
            start_hrs = self.slot_to_time(start)

            info = {
                "classroom_id": room,
                "start" : f"{day}_{start_hrs}",
                "end" : f"{day}_{end_hrs}",
                "lecturer_id" : lecturer,
                "duration": duration,
            }

            result[subject_id] = info
        
        # print(result)



    

    def _no_classroom_conflict(self, a, b, dur_a, dur_b):
        # logger.debug(f"{a}")
        # logger.debug(f"{b}")
        """
        a: tuple = Combination of day, slot and class  
        b: tuple = Combination of day, slot and class  
        """
        day_a, start_a, room_a = a
        day_b, start_b, room_b = b

        if day_a != day_b or room_a != room_b:
            return True

        # check overlap
        return (start_a + dur_a <= start_b) or (start_b + dur_b <= start_a)
    
    def _no_lecturer_conflict(self, a, b, lec_a, lec_b, dur_a, dur_b):
        """
        a, b = (day, start_time, room)
        """

        if lec_a != lec_b:
            return True
        
        day_a, start_a, _ = a
        day_b, start_b, _ = b

        if day_a != day_b:
            return True
        
        return (start_a + dur_a <= start_b) or (start_b + dur_b <= start_a) 

    # def _limit_classes_per_day(self, *assignments, max_per_day=3):
    #     day_counts = {}

    #     for a in assignments:
    #         day, start, room = a
    #         day_counts[day] = day_counts.get(day, 0) + 1

    #         if day_counts[day] > max_per_day:
    #             return False
            
    #     return True


    def _create_domains(self, subjects: List[Subject], classrooms: List[Classroom]):
        for subject in subjects:
            s_id = subject.subject_id
            duration = subject.duration
            assigned_class = subject.classroom_id

            logger.debug(f"ID: {s_id}, Duration: {duration} hrs, Classroom: {assigned_class.classroom_id}")

            domain = []

            for (day, start) in self.time_slots:
                end = start + duration

                if end <= int(self.hours_per_day * 2):
                    if assigned_class.classroom_id:
                        domain.append((day, start, assigned_class.classroom_id))
                    else:
                        for c in classrooms:
                            domain.append((day, start, c.classroom_id))
            random.shuffle(domain)
            self.problem.addVariable(s_id, domain)
            logger.info(f"Added domain: {domain}")

    def _add_constraints(self, subjects: List[Subject]):
        subject_ids = [sub.subject_id for sub in subjects]
        logger.debug(f"{subject_ids}")

        for i, sub_a in enumerate(subjects):
            for sub_b in subjects[i+1:]:
                
                # classroom conflict
                self.problem.addConstraint(
                    lambda a, b, sub_a=sub_a, sub_b=sub_b: self._no_classroom_conflict(a, b, sub_a.duration, sub_b.duration),
                    (sub_a.subject_id, sub_b.subject_id) 
                )
                # lecturer conflict
                self.problem.addConstraint(
                    lambda a, b, sub_a=sub_a, sub_b=sub_b: self._no_lecturer_conflict(
                        a, b, sub_a.lecturer_id, sub_b.lecturer_id, sub_a.duration, sub_b.duration
                        ),
                    (sub_a.subject_id, sub_b.subject_id) 
                )

                # prevent one-day clustering
                # self.problem.addConstraint(self._limit_classes_per_day, tuple(subject_ids))


TIMETABLE_CONFIG = {
    "subjects": subjects,
    "classrooms": available_classrooms,
    "model": CSPScheduler(hours_per_day=8, working_days=5),
}

timetable = Timetable(**TIMETABLE_CONFIG)
timetable.generate()

{'CS008': {'classroom_id': 'R301', 'start': 'Friday_09:00', 'end': 'Friday_13:00', 'lecturer_id': 'Dr. Aisha Patel', 'duration': 4.0}, 'CS010': {'classroom_id': 'Hall 1', 'start': 'Tuesday_10:30', 'end': 'Tuesday_14:30', 'lecturer_id': 'Dr. Aisha Patel', 'duration': 4.0}, 'CS014': {'classroom_id': 'Hall 1', 'start': 'Monday_07:30', 'end': 'Monday_11:30', 'lecturer_id': 'Prof. Michael Chen', 'duration': 4.0}, 'CS012': {'classroom_id': 'Hall 1', 'start': 'Thursday_07:00', 'end': 'Thursday_10:00', 'lecturer_id': 'Prof. Michael Chen', 'duration': 3.0}, 'CS004': {'classroom_id': 'Lab C', 'start': 'Tuesday_10:00', 'end': 'Tuesday_13:00', 'lecturer_id': 'Mr David', 'duration': 3.0}, 'CS006': {'classroom_id': 'R201', 'start': 'Wednesday_10:00', 'end': 'Wednesday_13:00', 'lecturer_id': 'Ms. Sarah', 'duration': 3.0}, 'CS005': {'classroom_id': 'Lab A', 'start': 'Wednesday_11:30', 'end': 'Wednesday_13:30', 'lecturer_id': 'Low Jing Xuan', 'duration': 2.0}, 'CS003': {'classroom_id': 'R301', 'start':